# 🎬 Rasmlardan kino-video (Wan 2.2, bepul GPU)

1. Yuqoridagi menyudan: **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all** ni bosing.
3. Google Drive ruxsatini so'rasa, **Allow** ni bosing. Kliplar Drive'ga saqlanadi va uzilib qolsa ham yo'qolmaydi.
4. Taxminan **1–1,5 soat** kuting. Oxirida `kino_final.mp4` avtomatik yuklab olinadi.

Agar Colab uzilib qolsa, **Run all** ni qayta bosing: tayyor kliplar qayta yaratilmaydi, ish to'xtagan joyidan davom etadi.

In [ ]:
#@title ⚙️ Sozlamalar
HD = False      #@param {type:"boolean"}
STEPS = 30      #@param {type:"integer"}
USE_DRIVE = True #@param {type:"boolean"}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 1) O'rnatish va fayllarni yuklash
!pip -q install -U "diffusers>=0.35" transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece
!rm -rf /content/ummon && git clone -q --depth 1 -b kino-video --filter=blob:none --sparse https://github.com/shoyadbeku/ummon /content/ummon && cd /content/ummon && git sparse-checkout set kino
import os
OUT = "/content/clips"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/kino_clips"
os.makedirs(OUT, exist_ok=True)
print("Kliplar papkasi:", OUT)

In [ ]:
#@title 2) Har bir rasmni videoga aylantirish (eng uzun qism)
args = f"--out {OUT} --steps {STEPS}" + (" --hd" if HD else "")
!cd /content/ummon/kino && python generate.py {args}

In [ ]:
#@title 3) Ovoz bilan birlashtirish, 1080x1920 ga kattalashtirish va yuklab olish
FINAL = "/content/kino_final.mp4"
!cd /content/ummon/kino && python assemble.py --clips {OUT} --out {FINAL}
if USE_DRIVE:
    !cp {FINAL} /content/drive/MyDrive/kino_final.mp4
from IPython.display import Video, display
display(Video(FINAL, embed=True, width=360))
from google.colab import files
files.download(FINAL)

### Bitta sahna yoqmadimi?
Drive'dagi `kino_clips/clipN.mp4` faylini o'chiring. `scenes.json` dagi promptni o'zgartirmoqchi bo'lsangiz, uni tahrirlang. Keyin 2- va 3-qadamni qayta ishga tushiring: faqat o'sha sahna qayta yaratiladi. Boshqa variant olish uchun `generate.py` ga `--seed 7` qo'shing.